# 29 - Réciprocité quadratique et symbole de Legendre

*Série Serre 100 — distillations exécutables* (EPIC #16334)

La **loi de réciprocité quadratique** est le cœur du chapitre I de *A Course
in Arithmetic* de Serre : elle relie, pour deux nombres premiers impairs
distincts `p` et `q`, la « résidu-ité » de `p` modulo `q` à celle de `q` modulo
`p`. Ce carnet **mesure** l'outil qui la porte — le **symbole de Legendre**
`(a/p)` — puis la vérifie de bout en bout :

- le symbole de Legendre `(a/p)` et la partition en résidus / non-résidus ;
- les **lois supplémentaires** `(2/p)` et `(-1/p)` ;
- la **loi de réciprocité** elle-même, `(p/q)·(q/p) = (-1)^{…}`.

Tout est exécuté et vérifié en Python ci-dessous. Le pendant formel (preuve par
le noyau Lean, `legendreSym.quadratic_reciprocity`) est un grain de suite de la série.


## Plan

1. Le symbole de Legendre `(a/p)` — définition par le critère d'Euler.
2. Les résidus quadratiques modulo `p`.
3. Les lois supplémentaires `(2/p)` et `(-1/p)`.
4. La loi de réciprocité quadratique.
5. Exercices.


## 1. Le symbole de Legendre

Pour un nombre premier impair `p` et un entier `a`, le **symbole de Legendre**
`(a/p)` est :

- `0` si `p` divise `a` ;
- `+1` si `a` est un **résidu quadratique** modulo `p` (il existe `x` avec
  `x² ≡ a [mod p]`) ;
- `-1` sinon.

Il se calcule sans chercher `x` grâce au **critère d'Euler** :
`(a/p) ≡ a^{(p-1)/2} [mod p]`, qui ne vaut que `0`, `1` ou `-1`.


In [1]:
def legendre(a, p):
    """Symbole de Legendre (a/p) par le critere d'Euler :
    a**((p-1)//2) mod p vaut 0, 1 ou p-1 (i.e. +1 ou -1)."""
    a %= p
    if a == 0:
        return 0
    r = pow(a, (p - 1) // 2, p)
    return 1 if r == 1 else -1

# Table du symbole de Legendre pour quelques petits premiers
for p in [3, 5, 7, 11, 13]:
    row = "  ".join(f"({a}/{p})={legendre(a, p):+d}" for a in range(1, p))
    print(f"p = {p:2d} :  {row}")


p =  3 :  (1/3)=+1  (2/3)=-1
p =  5 :  (1/5)=+1  (2/5)=-1  (3/5)=-1  (4/5)=+1
p =  7 :  (1/7)=+1  (2/7)=+1  (3/7)=-1  (4/7)=+1  (5/7)=-1  (6/7)=-1
p = 11 :  (1/11)=+1  (2/11)=-1  (3/11)=+1  (4/11)=+1  (5/11)=+1  (6/11)=-1  (7/11)=-1  (8/11)=-1  (9/11)=+1  (10/11)=-1
p = 13 :  (1/13)=+1  (2/13)=-1  (3/13)=+1  (4/13)=+1  (5/13)=-1  (6/13)=-1  (7/13)=-1  (8/13)=-1  (9/13)=+1  (10/13)=+1  (11/13)=-1  (12/13)=+1


### Lecture du résultat

Le symbole ne prend que les valeurs `+1` et `-1` hors du cas `p | a`, et il
**partitionne** les classes `1, …, p-1` en deux moitiés. Pour `p = 7`, la table
donne `2` comme résidu (`(2/7) = +1`) mais `3`, `5`, `6` comme non-résidus —
exactement la moitié. C'est cette oscillation `±1` que la réciprocité va relier
entre deux premiers.


## 2. Les résidus quadratiques modulo p

Les **résidus quadratiques** modulo `p` sont les `a` (dans `1, …, p-1`) pour
lesquels `(a/p) = +1`. Il y en a exactement `(p-1)/2` : les carrés
`1², 2², …` modulo `p` ne couvrent qu'une moitié des classes non nulles.


In [2]:
# Residus quadratiques modulo p : les a dans 1..p-1 avec (a/p) = +1
for p in [7, 11, 13]:
    res = [a for a in range(1, p) if legendre(a, p) == 1]
    print(f"p = {p:2d} : residues = {res}   "
          f"(|R| = {len(res)} = (p-1)/2 = {(p - 1) // 2})")


p =  7 : residues = [1, 2, 4]   (|R| = 3 = (p-1)/2 = 3)
p = 11 : residues = [1, 3, 4, 5, 9]   (|R| = 5 = (p-1)/2 = 5)
p = 13 : residues = [1, 3, 4, 9, 10, 12]   (|R| = 6 = (p-1)/2 = 6)


### Lecture du résultat

Pour `p = 7`, les résidus sont `[1, 2, 4]` ; pour `p = 11`, `[1, 3, 4, 5, 9]` ;
pour `p = 13`, `[1, 3, 4, 9, 10, 12]` — dans chaque cas exactement `(p-1)/2`
éléments. Le critère d'Euler retrouve ici, par un simple `pow(a, (p-1)//2, p)`,
la même partition qu'une énumération brute des carrés.


## 3. Les lois supplémentaires

Deux cas particuliers de la réciprocité se closent eux-mêmes :

- **`(-1/p)`** : vaut `+1` si `p ≡ 1 [mod 4]`, `-1` si `p ≡ 3 [mod 4]` —
  soit `(-1/p) = (-1)^{(p-1)/2}`.
- **`(2/p)`** : vaut `+1` si `p ≡ ±1 [mod 8]`, `-1` si `p ≡ ±3 [mod 8]` —
  soit `(2/p) = (-1)^{(p²-1)/8}`.

Elles « lèvent » les symboles dont la base est `-1` ou `2` sans faire appel à
la réciprocité proprement dite.


In [3]:
def is_prime(n):
    """n est-il premier ?"""
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True

odd_primes = [p for p in range(3, 41) if is_prime(p)]

# Loi (2/p)  :  (2/p) = (-1)^((p^2-1)/8)
bad2 = [p for p in odd_primes
        if legendre(2, p) != (1 if ((p * p - 1) // 8) % 2 == 0 else -1)]
# Loi (-1/p) :  (-1/p) = (-1)^((p-1)/2)
badm1 = [p for p in odd_primes
         if legendre(-1, p) != (1 if ((p - 1) // 2) % 2 == 0 else -1)]
print(f"Loi (2/p)  sur {len(odd_primes)} premiers impairs <= 40 : "
      f"contre-exemples = {bad2 if bad2 else 'aucun'}")
print(f"Loi (-1/p) sur {len(odd_primes)} premiers impairs <= 40 : "
      f"contre-exemples = {badm1 if badm1 else 'aucun'}")
# Apercu : (2/p) ne depend que de p modulo 8
print("  p mod 8 :  " + "   ".join(f"{p}->{p % 8}" for p in odd_primes))


Loi (2/p)  sur 11 premiers impairs <= 40 : contre-exemples = aucun
Loi (-1/p) sur 11 premiers impairs <= 40 : contre-exemples = aucun
  p mod 8 :  3->3   5->5   7->7   11->3   13->5   17->1   19->3   23->7   29->5   31->7   37->5


### Lecture du résultat

Les deux lois tiennent sur tous les premiers impairs `≤ 40` sans contre-exemple.
`(2/p)` ne dépend que de `p` **modulo 8** : `+1` pour `p ∈ {7, 17, 23, 31}`
(`≡ ±1 [mod 8]`) et `-1` pour `p ∈ {3, 5, 11, 13, 19, 29, 37}` (`≡ ±3 [mod 8]`) ;
`(-1/p) = +1` exactement quand `p ≡ 1 [mod 4]`.


## 4. La loi de réciprocité quadratique

Pour deux nombres premiers impairs **distincts** `p` et `q`, la loi de
**réciprocité quadratique** (Euler–Gauss) dit :

    (p/q) · (q/p) = (-1)^{ ((p-1)/2) · ((q-1)/2) }.

Le signe du membre de droite n'est `-1` que lorsque `p` **et** `q` sont tous
deux `≡ 3 [mod 4]` ; sinon il est `+1`. La loi permet de **retourner** le
symbole : pour un premier `a > p`, `(a/p)` s'exprime en `(p/a)` et un signe —
la base devient plus petite, et l'itération aboutit.


In [4]:
# Loi de reciprocite : pour p, q premiers impairs DISTINCTS,
#   (p/q) * (q/p) = (-1)^(((p-1)/2) * ((q-1)/2))
ps = [p for p in range(3, 41) if is_prime(p)]
bad, total = [], 0
for i in range(len(ps)):
    for j in range(i + 1, len(ps)):
        p, q = ps[i], ps[j]
        total += 1
        lhs = legendre(p, q) * legendre(q, p)
        exp = 1 if (((p - 1) // 2) * ((q - 1) // 2)) % 2 == 0 else -1
        if lhs != exp:
            bad.append((p, q, lhs, exp))
print(f"Verifie sur {total} paires de premiers impairs (p < q <= 40) : "
      f"{total - len(bad)}/{total} conformes.")
print("Contre-exemples :", bad if bad else "aucun")


Verifie sur 55 paires de premiers impairs (p < q <= 40) : 55/55 conformes.
Contre-exemples : aucun


### Lecture du résultat

La loi tient sur les `55` paires de premiers impairs `p < q ≤ 40` sans
contre-exemple. Le « retournement » se lit sur les échantillons : `(3/7) = -1`
et `(7/3) = +1`, de produit `-1` — le signe attendu, car `3` et `7` sont tous
deux `≡ 3 [mod 4]` ; alors que pour la paire `(5, 7)` le signe est `+1`
(`5 ≡ 1 [mod 4]`). C'est ce retournement itéré qui rend la réciprocité un
**algorithme** de décision de résidu-ité.


## 5. Exercices


In [5]:
# Exercice 1
# Ecrire `is_quadratic_residue(a, p)` : renvoyer True si `a` est un residu
# quadratique modulo `p` (il existe x avec x^2 == a [mod p]), False sinon.
# NE PAS utiliser le critere d'Euler : tester directement les x dans 1..p-1.
# Indice : chercher un x tel que (x * x) % p == a % p.
def is_quadratic_residue(a, p):
    # TODO etudiant
    return None
print("is_quadratic_residue(2, 7) =", is_quadratic_residue(2, 7))  # attendu : True
print("is_quadratic_residue(3, 7) =", is_quadratic_residue(3, 7))  # attendu : False


is_quadratic_residue(2, 7) = None
is_quadratic_residue(3, 7) = None


In [6]:
# Exercice 2
# Ecrire `non_residues(p)` : renvoyer la liste des NON-residus quadratiques
# dans 1..p-1, c'est-a-dire les a avec (a/p) = -1.
# Indice : [a for a in range(1, p) if legendre(a, p) == -1].
def non_residues(p):
    # TODO etudiant
    return None
print("non_residues(7) =", non_residues(7))  # attendu : [3, 5, 6]


non_residues(7) = None


In [7]:
# Exercice 3
# L'interet computatoire de la reciprocite est de « retourner » le symbole :
# pour un premier a > p, ecrire `flip_symbol(a, p)` qui renvoie le couple
# (signe, (p/a)), avec  (a/p) = signe * (p/a)  et
# signe = (-1)^(((a-1)/2) * ((p-1)/2)).
# Indice : signe = 1 si ((a-1)//2 * (p-1)//2) est pair, -1 sinon.
def flip_symbol(a, p):
    # TODO etudiant
    return None
print("flip_symbol(7, 3) =", flip_symbol(7, 3))  # (7/3) = signe * (3/7)


flip_symbol(7, 3) = None


## Conclusion

Le symbole de Legendre est un indicateur `±1` (ou `0`) de résidu-ité modulo un
premier ; le critère d'Euler le calcule en une puissance. Les lois
supplémentaires le closent pour les bases `-1` et `2` ; la **réciprocité
quadratique** le relie entre deux premiers et, en le retournant itérativement,
fournit un algorithme. Les trois volets sont mesurés ci-dessus sur des
dizaines de premiers ; leur preuve formelle, par le noyau Lean, fait l'objet
d'un carnet pendant de la série.


## Ressources

- J.-P. Serre, *A Course in Arithmetic* — ch. I (corps finis, § 3 : la loi de
  réciprocité quadratique, et son appendice qui en donne une seconde preuve),
  prolongé par le ch. III (symbole de Hilbert).
- Mathlib (Lean 4) : `legendreSym.quadratic_reciprocity`,
  `ZMod.exists_sq_eq_two_iff`, `ZMod.exists_sq_eq_prime_iff_of_mod_four_eq_one` —
  le pendant formel de ce carnet. Deux pièges de nommage : l'ordre des arguments
  est `legendreSym p a` (et non l'inverse), et les résidus quadratiques sont
  formalisés par `IsSquare` — le Mathlib épinglé ne définit aucun prédicat
  `IsQuadraticResidue`.

Retour à la série : [README de *Serre 100*](README.md)